# ADNI PHC: Amyloid PET 기반 Brain Age Gap 확장 분석

이 노트북은 앞선 MRI Brain Age Gap(MVP) 분석에서 계산한 BAG와 Amyloid PET 상태를 연결하는 **부차 분석**입니다. 연구 질문은 다음과 같습니다.

> MRI 촬영 시점과 가까운 Amyloid PET에서 양성인 참가자는, 음성인 참가자보다 더 높은 BAG를 보이는가?

BAG는 CN 구조 MRI로 학습한 정상 노화 기준으로부터의 구조적 편차입니다. Amyloid 양성은 PET 기반 병리 지표입니다. 따라서 이 분석은 구조적 뇌 노화 편차와 아밀로이드 병리의 **관찰적 연관성**을 평가하며, 개인 진단·질병 진행 예측·인과관계를 주장하지 않습니다.

## 분석 흐름과 사전 조건

```text
02 MVP 노트북 실행 및 BAG 저장
→ Amyloid PET의 QC 통과 기록 선택
→ RID별 MRI–PET 날짜 차이 계산
→ ±180일 이내 가장 가까운 PET 한 건 연결
→ 양성/음성별 EDA
→ 나이 상호작용을 포함한 공변량 보정 회귀
→ 기준 나이별 Amyloid 양성−음성 BAG 대비
→ 날짜 창 민감도 분석
```

먼저 `01_mri_brain_age_bag_mvp.ipynb`를 끝까지 실행해야 합니다. 이 노트북은 그 결과 파일인 `data/processed/phc_index_mri_scored_bag.csv`를 읽습니다. 뇌 나이 모델을 PET 분석에서 다시 학습하지 않으므로, PET 정보가 뇌 나이 모델에 섞이는 데이터 누수를 막을 수 있습니다.

## 0. 라이브러리와 분석 설정

`MAX_PET_MRI_DAYS=180`은 MRI와 PET가 충분히 가까운 시점에 측정되었다고 보는 사전 지정 날짜 창입니다. 날짜 창이 넓어지면 표본은 늘지만 MRI 구조와 PET 병리가 같은 상태를 반영한다는 가정은 약해집니다. 그래서 90일과 365일 창을 민감도 분석으로 함께 제시합니다.

PHC 데이터 사전에 따라 Amyloid PET 상태는 `0=Negative`, `1=Positive`이며, 영상 및 촬영 타이밍 QC가 모두 통과(`1`)한 PET만 사용합니다.

In [ ]:
%pip -q install pandas numpy matplotlib seaborn statsmodels scipy

from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr
import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm

warnings.filterwarnings('ignore', category=FutureWarning)
sns.set_theme(style='whitegrid', context='notebook')
MAX_PET_MRI_DAYS = 180
REFERENCE_AGES = [65, 75, 85]


## 1. 경로와 입력 파일 확인

원본 ADNI 자료는 Git에 올리지 않습니다. PET 확장 분석은 원본 PET 파일을 읽지만, 저장하는 결과는 표본 수·효과 추정치·집계 표·식별자 없는 그림으로 제한합니다.

`MRI_DATE`는 MVP에서 선택한 RID별 index MRI의 날짜이고, `PHC_SCANDATE`는 Amyloid PET 촬영일입니다. 날짜가 없거나 BAG가 없는 참가자는 정확한 시간 정렬 또는 결과 계산이 불가능하므로 제외합니다.

In [ ]:
def locate_project_root():
    start = Path.cwd().resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'data' / 'adni_all').exists():
            return candidate
    raise FileNotFoundError(
        '프로젝트 루트를 찾지 못했습니다. VS Code에서 brain-age-gap-adni 폴더를 연 뒤 실행하세요.'
    )

PROJECT_ROOT = locate_project_root()
print(f'Project root: {PROJECT_ROOT}')
DATA_DIR = PROJECT_ROOT / 'data'
ADNI_ROOT = DATA_DIR / 'adni_all'
# 분석 정의를 바꾸면 RUN_VERSION만 올립니다. 같은 정의를 재실행할 때는 그대로 둡니다.
ANALYSIS_ID = '02_amyloid_pet_bag'
RUN_VERSION = '20260820_v1'
RUN_ID = f'{ANALYSIS_ID}_{RUN_VERSION}'

PROCESSED_DIR = DATA_DIR / 'processed'
FIG_DIR = PROJECT_ROOT / 'results' / 'figures' / RUN_ID
TABLE_DIR = PROJECT_ROOT / 'results' / 'tables' / RUN_ID
for folder in [FIG_DIR, TABLE_DIR]:
    folder.mkdir(parents=True, exist_ok=True)
print(f'Result run ID: {RUN_ID}')

BAG_FILE = PROCESSED_DIR / 'phc_index_mri_scored_bag.csv'
PET_FILE = ADNI_ROOT / 'ADSP_PHC' / 'ADSP_PHC_PET_Amyloid_Simple_22Jan2026.csv'
assert BAG_FILE.exists(), f'MVP BAG 파일이 없습니다. 먼저 01 노트북을 실행하세요: {BAG_FILE}'
assert PET_FILE.exists(), f'Amyloid PET 파일이 없습니다: {PET_FILE}'

bag = pd.read_csv(BAG_FILE, low_memory=False)
pet = pd.read_csv(PET_FILE, low_memory=False)
required_bag = ['RID', 'MRI_DATE', 'BAG', 'AGE', 'diagnosis', 'sex', 'education', 'eTIV']
missing_bag = [column for column in required_bag if column not in bag.columns]
assert not missing_bag, f'MVP BAG 파일의 필수 열이 없습니다: {missing_bag}'
print(f'MVP BAG index MRI: {len(bag):,}명')
print(f'원본 Amyloid PET 기록: {len(pet):,}건')

## 2. PET 품질 관리와 MRI–PET 날짜 정렬

한 참가자에게 PET가 여러 번 있을 수 있으므로, 모든 PET를 무작정 결합하면 한 사람의 정보가 여러 번 반복됩니다. 이 노트북은 다음 규칙으로 한 사람당 PET 한 건만 고릅니다.

1. Amyloid 상태가 Negative 또는 Positive인 기록만 유지
2. PET image QC와 timing QC가 모두 Pass인 기록만 유지
3. index MRI와의 절대 날짜 차이가 180일 이하인 기록만 유지
4. 그중 MRI와 가장 가까운 PET 한 건만 선택

이 규칙은 PET 결과를 더 많이 확보하는 것보다 MRI BAG와 병리 상태의 시간적 대응을 우선합니다. 날짜 차이는 결과표에 기록하고, 90일·365일 창에서 결론이 유지되는지 확인합니다.

In [ ]:
def numeric_rid(frame):
    out = frame.copy()
    out['RID'] = pd.to_numeric(out['RID'], errors='coerce').astype('Int64')
    return out

bag = numeric_rid(bag)
pet = numeric_rid(pet)
bag['MRI_DATE'] = pd.to_datetime(bag['MRI_DATE'], errors='coerce')
pet['PET_DATE'] = pd.to_datetime(pet['PHC_SCANDATE'], errors='coerce')
pet['amyloid_status'] = pd.to_numeric(pet['PHC_AMYLOID_STATUS'], errors='coerce').map({0: 'Negative', 1: 'Positive'})
pet['PET_QC_IMAGE'] = pd.to_numeric(pet['PHC_QC_IMAGE'], errors='coerce')
pet['PET_QC_TIMING'] = pd.to_numeric(pet['PHC_QC_TIMING'], errors='coerce')

flow = []
flow.append({'stage': 'MVP index MRI with BAG', 'n': int(bag['RID'].nunique())})
flow.append({'stage': 'Raw Amyloid PET visits', 'n': int(len(pet))})
pet_qc = pet.loc[
    pet['amyloid_status'].isin(['Negative', 'Positive'])
    & pet['PET_DATE'].notna()
    & pet['PET_QC_IMAGE'].eq(1)
    & pet['PET_QC_TIMING'].eq(1)
].copy()
flow.append({'stage': 'PET with known status, date, image/timing QC pass', 'n': int(len(pet_qc))})

candidate_pairs = bag.merge(
    pet_qc[['RID', 'PET_DATE', 'amyloid_status', 'PHC_CENTILOIDS', 'PHC_TRACER']],
    on='RID', how='inner', validate='one_to_many'
).copy()
candidate_pairs['days_mri_to_pet'] = (candidate_pairs['PET_DATE'] - candidate_pairs['MRI_DATE']).dt.days
candidate_pairs['abs_days_mri_to_pet'] = candidate_pairs['days_mri_to_pet'].abs()
flow.append({'stage': f'MRI–PET candidate pairs within all dates', 'n': int(len(candidate_pairs))})

def nearest_pet_pair(pairs, max_days):
    eligible = pairs.loc[pairs['abs_days_mri_to_pet'].le(max_days)].copy()
    eligible = eligible.sort_values(['RID', 'abs_days_mri_to_pet', 'PET_DATE'])
    return eligible.drop_duplicates('RID', keep='first').copy()

aligned = nearest_pet_pair(candidate_pairs, MAX_PET_MRI_DAYS)
flow.append({'stage': f'Closest QC-pass PET within ±{MAX_PET_MRI_DAYS} days', 'n': int(len(aligned))})
flow_table = pd.DataFrame(flow)
display(flow_table)
flow_table.to_csv(TABLE_DIR / 'amyloid_pet_alignment_flow.csv', index=False)

assert aligned['RID'].is_unique, 'One PET record per RID should remain after nearest-date selection.'
assert aligned['amyloid_status'].isin(['Negative', 'Positive']).all()
print(f'PET-aligned analysis sample: {len(aligned):,}')
display(aligned[['days_mri_to_pet', 'abs_days_mri_to_pet']].describe().round(1))

## 3. EDA: Amyloid 상태와 분석 코호트 확인

Amyloid 양성/음성 집단은 나이와 진단 분포가 다를 수 있습니다. 예를 들어 양성 비율이 Dementia에서 높다면, 단순 BAG 평균 차이는 진단 단계의 차이를 반영할 수 있습니다. 따라서 이 셀에서 표본 수, 나이, BAG, 임상 진단, MRI–PET 날짜 차이를 먼저 확인하고, 다음 회귀 모형에서 진단·나이·성별·교육·eTIV를 함께 보정합니다.

Centiloid는 추적 설명용 연속 PET 값입니다. 추적자와 측정 조건이 다를 수 있으므로, 사전 정의된 Positive/Negative 상태를 주 분석 노출로 사용합니다.

In [ ]:
aligned['amyloid_status'] = pd.Categorical(aligned['amyloid_status'], ['Negative', 'Positive'])
status_summary = (aligned.groupby('amyloid_status', observed=False)
                  .agg(n=('RID', 'size'), age_mean=('AGE', 'mean'), BAG_mean=('BAG', 'mean'),
                       BAG_sd=('BAG', 'std'), days_mri_pet_median=('abs_days_mri_to_pet', 'median'))
                  .round(2))
status_by_diagnosis = pd.crosstab(aligned['diagnosis'], aligned['amyloid_status'], dropna=False)
display(status_summary)
display(status_by_diagnosis)
status_summary.to_csv(TABLE_DIR / 'amyloid_pet_bag_summary_by_status.csv')
status_by_diagnosis.to_csv(TABLE_DIR / 'amyloid_status_by_diagnosis.csv')

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
sns.boxplot(data=aligned, x='amyloid_status', y='BAG', palette='Set2', showfliers=False, ax=axes[0])
sns.stripplot(data=aligned, x='amyloid_status', y='BAG', color='0.25', alpha=0.35, size=3, ax=axes[0])
axes[0].axhline(0, color='black', lw=1)
axes[0].set(title='BAG by Amyloid PET status', xlabel='Amyloid status', ylabel='Bias-corrected BAG (years)')
sns.histplot(data=aligned, x='abs_days_mri_to_pet', hue='amyloid_status', bins=20, stat='density', common_norm=False, ax=axes[1])
axes[1].set(title='MRI–PET alignment window', xlabel='Absolute MRI–PET date difference (days)')
plt.tight_layout(); plt.savefig(FIG_DIR / 'amyloid_pet_eda.png', dpi=300); plt.show()

## 4. 주 분석: Amyloid 상태 × 나이

주 모형은 다음과 같습니다.

```text
BAG ~ amyloid_status × AGE + diagnosis + sex + education + eTIV
```

`amyloid_status × AGE`를 포함하는 이유는 현재 MVP에서 진단군 BAG 차이가 나이에 따라 달라졌기 때문입니다. Amyloid 양성/음성의 BAG 차이도 나이에 따라 달라질 수 있으므로, 이 가정을 직접 검정합니다.

모형은 임상 진단을 보정하므로, 결과는 ‘같은 진단 단계·나이·성별·교육·eTIV 조건에서 Amyloid 상태와 BAG가 연관되는가’에 대한 추정입니다. 이는 Amyloid가 BAG를 일으킨다는 의미가 아닙니다.

In [ ]:
primary_df = aligned.dropna(subset=['BAG', 'AGE', 'diagnosis', 'sex', 'education', 'eTIV', 'amyloid_status']).copy()
assert primary_df['amyloid_status'].nunique() == 2, 'Both Amyloid groups are required for comparison.'

amyloid_fit = smf.ols(
    'BAG ~ C(amyloid_status) * AGE + C(diagnosis) + C(sex) + education + eTIV',
    data=primary_df
).fit()
amyloid_ancova = anova_lm(amyloid_fit, typ=2, robust='hc3')
display(amyloid_ancova)
amyloid_ancova.to_csv(TABLE_DIR / 'amyloid_pet_bag_ancova_hc3.csv')

robust_amyloid = amyloid_fit.get_robustcov_results(cov_type='HC3')
coef_table = pd.DataFrame({
    'term': robust_amyloid.model.exog_names, 'estimate': robust_amyloid.params,
    'std_error_HC3': robust_amyloid.bse, 'p_value': robust_amyloid.pvalues,
    'ci_low': robust_amyloid.conf_int()[:, 0], 'ci_high': robust_amyloid.conf_int()[:, 1],
})
display(coef_table.round(4))
coef_table.to_csv(TABLE_DIR / 'amyloid_pet_bag_coefficients_hc3.csv', index=False)

interaction_p = amyloid_ancova.loc['C(amyloid_status):AGE', 'PR(>F)']
print(f'Amyloid status × AGE interaction p-value: {interaction_p:.3e}')
print(f'Primary PET regression sample: {len(primary_df):,}')

## 5. 기준 나이별 Amyloid 양성−음성 BAG 대비

상호작용이 유의하면 Amyloid 상태의 단일 평균 차이를 주 결과로 사용하지 않습니다. 대신 65세·75세·85세에서 Positive와 Negative의 조정 BAG 차이를 제시합니다. 상호작용이 유의하지 않더라도 이 표는 효과가 나이에 따라 얼마나 안정적인지 보여 주는 보조 자료가 됩니다.

회귀 대비는 HC3 robust standard error와 95% 신뢰구간으로 계산합니다.

In [ ]:
term_index = {term: i for i, term in enumerate(robust_amyloid.model.exog_names)}

def amyloid_contrast_at_age(reference_age):
    main_term = 'C(amyloid_status)[T.Positive]'
    interaction_term = 'C(amyloid_status)[T.Positive]:AGE'
    missing_terms = [term for term in [main_term, interaction_term] if term not in term_index]
    if missing_terms:
        raise KeyError(f'Expected Amyloid model terms are missing: {missing_terms}')
    contrast = np.zeros(len(term_index))
    contrast[term_index[main_term]] = 1.0
    contrast[term_index[interaction_term]] = float(reference_age)
    result = robust_amyloid.t_test(contrast)
    ci_low, ci_high = np.asarray(result.conf_int(alpha=0.05)).ravel()
    return {
        'reference_age': reference_age, 'contrast': 'Amyloid Positive vs Negative',
        'BAG_difference_years': float(np.asarray(result.effect).squeeze()),
        'std_error_HC3': float(np.asarray(result.sd).squeeze()),
        'p_value': float(np.asarray(result.pvalue).squeeze()),
        'ci_low': float(ci_low), 'ci_high': float(ci_high),
    }

age_specific_contrasts = pd.DataFrame([amyloid_contrast_at_age(age) for age in REFERENCE_AGES])
display(age_specific_contrasts.round(4))
age_specific_contrasts.to_csv(TABLE_DIR / 'amyloid_pet_age_specific_bag_contrasts_hc3.csv', index=False)

reference_diagnosis = primary_df['diagnosis'].mode().iat[0]
reference_sex = 'Female' if 'Female' in primary_df['sex'].dropna().unique() else primary_df['sex'].mode().iat[0]
age_grid = np.linspace(primary_df['AGE'].quantile(0.02), primary_df['AGE'].quantile(0.98), 100)
prediction_frames = []
for status in ['Negative', 'Positive']:
    new_data = pd.DataFrame({
        'amyloid_status': status, 'AGE': age_grid, 'diagnosis': reference_diagnosis,
        'sex': reference_sex, 'education': primary_df['education'].mean(), 'eTIV': primary_df['eTIV'].mean(),
    })
    new_data['predicted_BAG'] = amyloid_fit.get_prediction(new_data).summary_frame()['mean'].to_numpy()
    prediction_frames.append(new_data)
trajectory = pd.concat(prediction_frames, ignore_index=True)
plt.figure(figsize=(8, 5))
sns.lineplot(data=trajectory, x='AGE', y='predicted_BAG', hue='amyloid_status', linewidth=2.5)
plt.axhline(0, color='black', lw=1)
plt.ylabel('Adjusted predicted BAG (years)')
plt.title('Amyloid PET status and BAG across age')
plt.tight_layout(); plt.savefig(FIG_DIR / 'amyloid_pet_bag_age_interaction.png', dpi=300); plt.show()

## 6. 날짜 창 민감도 분석

이 분석은 결과의 방향이 선택한 날짜 창 하나에만 의존하는지 점검합니다. 동일한 QC·가장 가까운 PET 규칙을 적용하되, ±90일·±180일·±365일에서 표본 수와 Amyloid 양성−음성의 단순 BAG 평균 차이를 요약합니다.

이 표는 주 분석을 대체하지 않습니다. 주 분석의 추론은 사전 지정 ±180일 정렬과 공변량 보정 회귀에서 얻습니다.

In [ ]:
sensitivity_rows = []
for window in [90, 180, 365]:
    sample = nearest_pet_pair(candidate_pairs, window)
    group_means = sample.groupby('amyloid_status', observed=False)['BAG'].mean()
    sensitivity_rows.append({
        'max_alignment_days': window, 'n': len(sample),
        'negative_n': int(sample['amyloid_status'].eq('Negative').sum()),
        'positive_n': int(sample['amyloid_status'].eq('Positive').sum()),
        'negative_BAG_mean': group_means.get('Negative', np.nan),
        'positive_BAG_mean': group_means.get('Positive', np.nan),
        'unadjusted_positive_minus_negative_BAG': group_means.get('Positive', np.nan) - group_means.get('Negative', np.nan),
    })

alignment_sensitivity = pd.DataFrame(sensitivity_rows)
display(alignment_sensitivity.round(3))
alignment_sensitivity.to_csv(TABLE_DIR / 'amyloid_pet_alignment_window_sensitivity.csv', index=False)


## 결과를 해석하는 틀과 다음 확장

발표와 포트폴리오에서는 아래 순서로 결과를 설명합니다.

1. **정렬 품질**: ‘MRI–PET 날짜 차이 ±180일 규칙으로 총 __명을 연결했고, 중앙 날짜 차이는 __일이었다.’
2. **주 결과**: ‘진단·나이·성별·교육·eTIV를 보정한 Amyloid 상태×나이 분석에서 __을 확인했다.’
3. **효과 크기**: ‘75세에서 Amyloid 양성군의 조정 BAG는 음성군보다 __년 높거나 낮았다(95% CI __–__).’
4. **한계**: ‘ADNI 관찰자료와 한 시점 근접 정렬만으로 Amyloid 축적이 BAG를 유발한다고 결론낼 수 없다.’

PET 확장 후에는 MRI ROI의 normative Z-score, eTIV·ROI 집합·날짜 창 민감도 분석을 완료합니다. 그 결과가 안정적일 때 Desikan-Killiany 68개 피질 영역의 FDR 보정 BAG 연관성을 fsaverage5 표면에 시각화합니다. 뇌 표면 그림은 개인 MRI나 인과적 병변 지도가 아니라, 모델에 사용된 ROI와 BAG의 집단 수준 연관성 지도입니다.